# Tutoriel InfluxDB


## Création du client

In [2]:
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
from datetime import datetime

url = "http://influxdb2:8086"
token = open("/run/secrets/influxdb2-admin-token").read().strip()
org = "docs"

client = InfluxDBClient(url=url, token=token, org=org)
#write_api = client.write_api(write_options=SYNCHRONOUS)

# point = (
#     Point("temperature")
#     .tag("room", "salon")
#     .field("value", 23.5)
#     .time(datetime.utcnow(), WritePrecision.NS)
# )

# write_api.write(bucket=bucket, org=org, record=point)

# print("Data written")

## Créaction d'un bucket et importation des données

Organisation des données : 
* **Bucket** : où sont stockées les données
    * **Measurement** : groupe de données
        * Tags : Pairs clé-valeur qui ne changent pas souvent, metadata - ex : location
        * Fields : Pairs clé-valeur qui changent au cours du temps - ex : temperature
        * Timestamp : permet d'indexer les valeurs au cours du temps

In [ ]:
from influxdb_client import BucketsApi

buckets_api = client.buckets_api()

buckets_api.create_bucket(
    bucket_name="air-sensor",
    org=org
)

{'created_at': datetime.datetime(2026, 3, 19, 10, 23, 3, 897862, tzinfo=tzlocal()),
 'description': None,
 'id': 'f6a236bec164443a',
 'labels': [],
 'links': {'_self': '/api/v2/buckets/f6a236bec164443a',
           'labels': '/api/v2/buckets/f6a236bec164443a/labels',
           'members': '/api/v2/buckets/f6a236bec164443a/members',
           'org': '/api/v2/orgs/61e32bf9807aa061',
           'owners': '/api/v2/buckets/f6a236bec164443a/owners',
           'write': '/api/v2/write?org=61e32bf9807aa061&bucket=f6a236bec164443a'},
 'name': 'air_sensor',
 'org_id': '61e32bf9807aa061',
 'retention_rules': [{'every_seconds': 0,
                      'shard_group_duration_seconds': 604800,
                      'type': 'expire'}],
 'rp': '0',
 'schema_type': None,
 'type': 'user',
 'updated_at': datetime.datetime(2026, 3, 19, 10, 23, 3, 897862, tzinfo=tzlocal())}

Pour importer les données, on utilise une requête Flux. C'est le langage utilisé dans influxDB pour intéragir avec les données.

In [9]:
query = '''
import "influxdata/influxdb/sample"

sample.data(set: "airSensor")
    |> to(bucket: "air_sensor")
'''

# option task = {
#   name: "Collect air sensor sample data",
#   every: 15m,
# }

client.query_api().query(query)

[<FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>,
 <FluxTable: 7 columns, 361 records>]

Une requête Flux est composée de  :
* `from` pour définir la source
* Pipe-forward operator `|>` pour implémenter des fonctions sur l'entrée précédente

La requête se base sur une succession d'opération qui renvoie finalement la transformation de l'entrée jusqu'à la sortie recherchée.

In [8]:
query = '''
from (bucket: "air_sensor")
    |> range(start : 0)
    |> filter(fn: (r) => r._measurement == "airSensors")
    |> filter(fn: (r) => r._field == "temperature")
    |> mean()
    |> yield(name: "_results")
'''

tables = client.query_api().query(query)

for table in tables:
    for record in table.records:
        print(record.values)

{'result': '_results', 'table': 0, '_start': datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), '_stop': datetime.datetime(2026, 3, 19, 12, 24, 59, 71575, tzinfo=datetime.timezone.utc), '_value': 71.5034809868646, '_field': 'temperature', '_measurement': 'airSensors', 'sensor_id': 'TLM0100'}
{'result': '_results', 'table': 1, '_start': datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), '_stop': datetime.datetime(2026, 3, 19, 12, 24, 59, 71575, tzinfo=datetime.timezone.utc), '_value': 71.16489727446739, '_field': 'temperature', '_measurement': 'airSensors', 'sensor_id': 'TLM0101'}
{'result': '_results', 'table': 2, '_start': datetime.datetime(1970, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), '_stop': datetime.datetime(2026, 3, 19, 12, 24, 59, 71575, tzinfo=datetime.timezone.utc), '_value': 72.37734814056294, '_field': 'temperature', '_measurement': 'airSensors', 'sensor_id': 'TLM0102'}
{'result': '_results', 'table': 3, '_start': datetime.datetime(1970, 1,

In [10]:
from influxdb_client import BucketsApi

buckets_api = client.buckets_api()

buckets_api.create_bucket(
    bucket_name="weather",
    org=org
)

{'created_at': datetime.datetime(2026, 3, 19, 10, 40, 39, 388423, tzinfo=tzlocal()),
 'description': None,
 'id': '890c8a4f26178221',
 'labels': [],
 'links': {'_self': '/api/v2/buckets/890c8a4f26178221',
           'labels': '/api/v2/buckets/890c8a4f26178221/labels',
           'members': '/api/v2/buckets/890c8a4f26178221/members',
           'org': '/api/v2/orgs/61e32bf9807aa061',
           'owners': '/api/v2/buckets/890c8a4f26178221/owners',
           'write': '/api/v2/write?org=61e32bf9807aa061&bucket=890c8a4f26178221'},
 'name': 'weather',
 'org_id': '61e32bf9807aa061',
 'retention_rules': [{'every_seconds': 0,
                      'shard_group_duration_seconds': 604800,
                      'type': 'expire'}],
 'rp': '0',
 'schema_type': None,
 'type': 'user',
 'updated_at': datetime.datetime(2026, 3, 19, 10, 40, 39, 388423, tzinfo=tzlocal())}

In [11]:
query = '''
import "influxdata/influxdb/sample"

sample.data(set: "noaa")
    |> to(bucket: "weather")
'''

# option task = {
#   name: "Collect NOAA NDBC sample data",
#   every: 15m,
# }

client.query_api().query(query)

[<FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 <FluxTable: 11 columns, 1 records>,
 

In [2]:
data = "temperature,room=cuisine value=21.5"

write_api.write(bucket="home", org="docs", record=data)